# 01 — Pandas: Series, DataFrame, Cleaning, GroupBy, Merge, Pivot

**Overview:** load a small synthetic support-ticket dataset and work through filtering, missing
values, duplicates, groupby/aggregation, merge/concat, and pivot/melt.

**Required libraries:** `pandas`. If `data/support_tickets.csv` doesn't exist yet, this notebook
regenerates it deterministically (no external download).

Connects to: `02_numpy` (a DataFrame column is array-backed). Leads to: `02_polars.ipynb`
(the same pipeline, Polars-style).

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **ML** | Machine Learning | Learning patterns from data to make predictions, instead of hand-coding rules. |
| **API** | Application Programming Interface | The set of functions/classes a library exposes for you to call. |

## 0. Environment & data

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)


DATA_PATH = Path("data/support_tickets.csv")

def generate_support_tickets_csv(path, n=300, seed=42):
    """Self-contained, deterministic synthetic dataset -- no external script or
    download needed, so this cell works the same in a fresh Colab runtime as it
    does locally."""
    import csv, random
    rng = random.Random(seed)
    categories = ["billing", "login", "performance", "bug", None]
    priorities = ["low", "medium", "high"]
    rows = []
    for i in range(n):
        rows.append({
            "ticket_id": i,
            "user_id": rng.randint(1, 50),
            "category": rng.choice(categories),
            "priority": rng.choice(priorities),
            "resolved": rng.choice([True, False]),
            "response_time_minutes": round(rng.uniform(5, 500), 1),
        })
    rows.extend(rows[:10])   # deterministic duplicates, used later to demo dedup handling
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

if not DATA_PATH.exists():
    generate_support_tickets_csv(DATA_PATH)

df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
print(df)

Python: 3.14.6
pandas: 3.0.6
shape: (310, 6)
     ticket_id  user_id     category priority  resolved  response_time_minutes
0            0       41      billing      low     False                  126.2
1            1        9      billing     high      True                  297.3
2            2        3      billing      low      True                  120.2
3            3       39      billing     high      True                  359.4
4            4       45          NaN   medium      True                  227.4
..         ...      ...          ...      ...       ...                    ...
305          5       18      billing      low     False                  173.4
306          6       10        login   medium      True                   50.9
307          7        7  performance   medium     False                  404.5
308          8       47          bug     high      True                  486.7
309          9       25      billing     high     False                  415.6

[310 r

## 1. Series vs DataFrame

💡 **Key Idea:** a `DataFrame` column is a `Series` — a 1D labeled array, conceptually a NumPy
array plus an index.

In [2]:
category_series = df["category"]
print(type(category_series))
print("Series shape:", category_series.shape, "DataFrame shape:", df.shape)
assert isinstance(category_series, pd.Series)
assert category_series.shape == (df.shape[0],)
print(pd.Series([1, 2, 3,100, 1000, 1000, 1000]).value_counts())


<class 'pandas.Series'>
Series shape: (310,) DataFrame shape: (310, 6)
1000    3
1       1
2       1
3       1
100     1
Name: count, dtype: int64


## 2. Filtering and sorting

In [ ]:
print("shape:", df)
print("priority:", df["priority"])

high_priority = df[df["priority"] == "high"]

print("high priority tickets:*****", len(high_priority), "/", len(df))

slow_tickets = df.sort_values("response_time_minutes", ascending=False).head(5)
print(slow_tickets[["ticket_id", "response_time_minutes"]])

assert (high_priority["priority"] == "high").all()


shape:      ticket_id  user_id     category priority  resolved  response_time_minutes
0            0       41      billing      low     False                  126.2
1            1        9      billing     high      True                  297.3
2            2        3      billing      low      True                  120.2
3            3       39      billing     high      True                  359.4
4            4       45          NaN   medium      True                  227.4
..         ...      ...          ...      ...       ...                    ...
305          5       18      billing      low     False                  173.4
306          6       10        login   medium      True                   50.9
307          7        7  performance   medium     False                  404.5
308          8       47          bug     high      True                  486.7
309          9       25      billing     high     False                  415.6

[310 rows x 6 columns]
priority: 0         l

## 3. Missing values and duplicates

⚠️ **Common Mistake:** aggregating before deduplicating silently double-counts rows.

In [4]:
#print("missing category count BEFORE cleaning:", df["category"].isna().sum())
print("duplicate rows BEFORE cleaning:", df.duplicated())

rows_before = len(df)
df_clean = df.drop_duplicates().copy()
df_clean["category"] = df_clean["category"].fillna("unknown")
rows_after = len(df_clean)

print(f"rows before dedup: {rows_before}, after: {rows_after}")
print("missing category count AFTER fillna:", df_clean["category"].isna().sum())

assert rows_after < rows_before          # we injected duplicates on purpose
assert df_clean["category"].isna().sum() == 0


duplicate rows BEFORE cleaning: 0      False
1      False
2      False
3      False
4      False
       ...  
305     True
306     True
307     True
308     True
309     True
Length: 310, dtype: bool
rows before dedup: 310, after: 300
missing category count AFTER fillna: 0


## 4. groupby + aggregation

🎯 **Interview Tip:** groupby+agg is the single most common tabular ML-prep operation.

In [5]:
tickets_per_category = df_clean.groupby("category").agg(
    n_tickets=("ticket_id", "count"),
    avg_response_time=("response_time_minutes", "mean"),
    resolved_rate=("resolved", "mean"),
).reset_index().sort_values("n_tickets", ascending=False)

print(tickets_per_category)
assert "n_tickets" in tickets_per_category.columns
assert tickets_per_category["n_tickets"].sum() == len(df_clean)


      category  n_tickets  avg_response_time  resolved_rate
4      unknown         71         255.056338       0.478873
3  performance         61         246.924590       0.459016
2        login         58         281.262069       0.568966
1          bug         57         275.931579       0.421053
0      billing         53         271.615094       0.584906


## 5. merge / concat

In [6]:
# A small lookup table mapping user_id -> a region, to demonstrate merge.
import random
rng = random.Random(0)
user_ids = sorted(df_clean["user_id"].unique())
region_lookup = pd.DataFrame({
    "user_id": user_ids,
    "region": [rng.choice(["us", "eu", "apac"]) for _ in user_ids],
})

merged = df_clean.merge(region_lookup, on="user_id", how="left")
print("merged shape:", merged.shape, "(same row count as df_clean, one new column)")
assert merged.shape[0] == df_clean.shape[0]
assert "region" in merged.columns

more_tickets = df_clean.sample(3, random_state=42)
concatenated = pd.concat([df_clean, more_tickets], ignore_index=True)
print("concat shape:", concatenated.shape, "== original + 3 sampled rows")
assert concatenated.shape[0] == df_clean.shape[0] + 3


merged shape: (300, 7) (same row count as df_clean, one new column)
concat shape: (303, 6) == original + 3 sampled rows


## 6. pivot / melt

In [7]:
pivot = merged.pivot_table(index="region", columns="priority", values="ticket_id", aggfunc="count", fill_value=0)
print("pivot (region x priority counts):\n", pivot)

melted = pivot.reset_index().melt(id_vars="region", var_name="priority", value_name="count")
print("\nmelted back to long form:\n", melted.head())

assert pivot.values.sum() == len(merged)


pivot (region x priority counts):
 priority  high  low  medium
region                     
apac        31   33      37
eu          46   40      43
us          27   23      20

melted back to long form:
   region priority  count
0   apac     high     31
1     eu     high     46
2     us     high     27
3   apac      low     33
4     eu      low     40


## 7. apply/map (last resort)

⚠️ **Common Mistake:** reaching for `.apply()` with a Python function when a vectorized
expression does the same thing faster.

In [8]:
import time

# Slow: .apply with a Python function
start = time.perf_counter()
_ = df_clean["response_time_minutes"].apply(lambda x: x / 60.0)
apply_time = time.perf_counter() - start

# Fast: vectorized
start = time.perf_counter()
_ = df_clean["response_time_minutes"] / 60.0
vector_time = time.perf_counter() - start

print(f".apply(): {apply_time*1000:.3f} ms, vectorized: {vector_time*1000:.3f} ms")
print(f"vectorized is {apply_time / max(vector_time, 1e-9):.1f}x faster on this tiny dataset "
      f"(the gap grows much larger on bigger data)")


.apply(): 0.245 ms, vectorized: 0.116 ms
vectorized is 2.1x faster on this tiny dataset (the gap grows much larger on bigger data)


## 8. Connection to what's next

The exact same `df_clean` -> groupby -> merge pipeline gets reproduced with Polars' expression
API (and lazy execution) in `02_polars.ipynb`.

## 9. Interview questions
1. What's the relationship between a `Series` and a `DataFrame`?
2. Why should you deduplicate before aggregating, not after?
3. When would `merge` produce more rows than either input DataFrame?
4. What's the difference between `pivot_table` and `melt`?
5. Why is `.apply()` usually slower than a vectorized expression?

## 10. Exercises
1. Compute the resolved rate per priority level instead of per category.
2. Find the 3 users with the most tickets using `groupby` + `sort_values`.
3. Pivot the cleaned data by `(priority, resolved)` and interpret the resulting table.

## 11. Summary

✅ Pandas' `DataFrame`/`Series`, filtering, groupby/agg, merge/concat, and pivot/melt cover the
core tabular-cleaning vocabulary. Next: `02_polars.ipynb` — the same ideas, with lazy, parallel
execution.